# Preparing Data (Label Encoding)

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

htl_df = pd.read_csv("cleaned_hotel_bookings.csv")

htl_df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,Overall_stay
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03,2


In [10]:
sect_cols = ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']

lab_en = LabelEncoder()

for col in sect_cols:
    htl_df[col] = lab_en.fit_transform(htl_df[col])

print(htl_df.head())

   hotel  is_canceled  lead_time  arrival_date_year  arrival_date_month  \
0      1            0        342               2015                   5   
1      1            0        737               2015                   5   
2      1            0          7               2015                   5   
3      1            0         13               2015                   5   
4      1            0         14               2015                   5   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  agent  company  \
0                        0                     0       2  ...    0.0      0.0   
1                        0            

# Splitting the Data

In [11]:
from sklearn.model_selection import train_test_split

# Separate the target (is_canceled) from the features
X = htl_df.drop('is_canceled', axis=1) # All columns except the target
y = htl_df['is_canceled']             # The target variable

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (69916, 32)
Testing shape: (17480, 32)


# Train the Model

In [12]:
# 1. Drop columns that are not useful for prediction (like dates or text status)
cols_to_drop = ['reservation_status', 'reservation_status_date']
X = X.drop(columns=cols_to_drop)

# 2. Make sure all data is numeric
# This converts any remaining object/string columns to numbers
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# 3. Now re-split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Try training again
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000) # Increased iterations to help it converge
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


/Users/prithvisadanand/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Evaluate the Model

In [13]:
from sklearn.metrics import accuracy_score, classification_report

# Use the model to predict on the test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Show the full report
print(classification_report(y_test, y_pred))

Accuracy: 0.77
              precision    recall  f1-score   support

           0       0.79      0.92      0.85     12733
           1       0.63      0.36      0.46      4747

    accuracy                           0.77     17480
   macro avg       0.71      0.64      0.66     17480
weighted avg       0.75      0.77      0.75     17480



# Random Forest Model

In [14]:
from sklearn.ensemble import RandomForestClassifier

# Create and train a Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_pred = rf_model.predict(X_test)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred):.2f}")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.85
              precision    recall  f1-score   support

           0       0.88      0.93      0.90     12733
           1       0.77      0.65      0.70      4747

    accuracy                           0.85     17480
   macro avg       0.82      0.79      0.80     17480
weighted avg       0.85      0.85      0.85     17480



# Identifying Feature Importance

In [18]:
import matplotlib.pyplot as plt

# Get the importance of each feature from our Random Forest model
key_factors = rf_model.feature_importances_
feature_names = X.columns

# Create a dataframe to display the results nicely
feature_key_htl_df = pd.DataFrame({'Feature': feature_names, 'Importance': key_factors})
feature_key_htl_df = feature_key_htl_df.sort_values(by='Importance', ascending=False)

# Display the top 10 most important features
print(feature_key_htl_df.head(10))

                      Feature  Importance
1                   lead_time    0.136874
12                    country    0.101937
26                        adr    0.091184
5   arrival_date_day_of_month    0.067347
4    arrival_date_week_number    0.063476
28  total_of_special_requests    0.059437
22                      agent    0.057111
13             market_segment    0.047919
29               Overall_stay    0.039514
7        stays_in_week_nights    0.035222
